## Data Pipeline Overview

This script uses the PRAW library to connect to Reddit and fetch recent posts from the subreddits "anxiety", "depression", and "adhd". It retrieves posts in batches, filters out duplicates, and only includes posts from the last 10 years. For each post, it extracts details like the author's name, creation time, score, title, and selftext content. The script handles potential API errors with retry logic and pauses to avoid rate limits. Once all data is collected, it saves the results in `reddit_data.csv` for further acquistion.

This script processes Reddit post data by cleaning, tokenizing, and lemmatizing the text content from `reddit_data.csv`. It first ensures necessary NLP resources are available using `nltk` and `spaCy`, then tokenizes the selftext (main post content), removes stopwords and punctuation, and quotes each word. It also lemmatizes the cleaned tokens to reduce words to their base forms. If a previously processed file (`merged_output_tokenized.csv`) exists, it merges the new data while avoiding duplicates. Finally, it saves the cleaned and merged dataset back to a CSV file.



In [ ]:
!pip install praw
!pip install schedule

In [ ]:
import praw
import pandas as pd
import time
import os
from datetime import datetime, timedelta

reddit = praw.Reddit(
    client_id="TIyXs6Cidt4b_Iut84v0jw",
    client_secret="r7JPzh046bVXuBqKlQbmhVAniwev7Q",
    user_agent="macOS:sample testing:v1.0.0 (by u/ostwalaman)"
)

subreddits = ["anxiety", "depression", "adhd"]

posts_per_batch = 10000

def fetch_reddit_data():
    print("Starting Reddit data fetch...")
    data = []
    unique_post_ids = set()
    one_year_ago = datetime.utcnow() - timedelta(days=3650)
    one_year_ago_timestamp = int(one_year_ago.timestamp())

    for subreddit_name in subreddits:
        subreddit = reddit.subreddit(subreddit_name)
        last_post_time = None

        while True:
            try:
                posts = subreddit.new(limit=posts_per_batch, params={"before": last_post_time} if last_post_time else {})
                posts_fetched = 0

                for post in posts:
                    if post.created_utc < one_year_ago_timestamp:
                        break
                    if post.id in unique_post_ids:
                        continue
                    unique_post_ids.add(post.id)

                    created_datetime = datetime.utcfromtimestamp(post.created_utc).strftime('%Y-%m-%d %H:%M:%S')

                    data.append({
                        "Author": post.author.name if post.author else None,
                        "Created_DateTime": created_datetime,
                        "Score": post.score,
                        "Selftext": post.selftext,
                        "Subreddit": post.subreddit.display_name,
                        "Title": post.title
                    })
                    last_post_time = post.created_utc
                    posts_fetched += 1

                if posts_fetched == 0:
                    break
            except Exception as e:
                print(f"Error fetching posts from r/{subreddit_name}: {e}")
                time.sleep(5)

            time.sleep(1)

    df = pd.DataFrame(data)

    df.to_csv("reddit_data.csv", index=False)
print("Data saved as reddit_data.csv")

In [ ]:
fetch_reddit_data()

In [ ]:
pd.read_csv("reddit_data.csv")

In [ ]:
import os
import pandas as pd
import nltk
import spacy
import time
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import csv

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

nlp = spacy.load("en_core_web_sm")

scraped_file = "/content/reddit_data.csv"  # New fetched data
processed_output_file = "/content/merged_output_tokenized.csv"  # Final processed output

def clean_and_tokenize(text):
    """Tokenize and clean text by removing stopwords and punctuation, and quote each word."""
    if pd.isna(text):
        return ""
    text = text.lower()
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [f'"{word}"' for word in tokens if word.isalpha() and word not in stop_words]  # Quote each word
    return " ".join(tokens)

def lemmatize_text(text):
    """Perform lemmatization on the text using spaCy and quote each lemmatized word."""
    if isinstance(text, str):
        doc = nlp(text)
        return " ".join([f'"{token.lemma_}"' for token in doc])  # Quote each lemmatized word
    return text

def clean_process_and_merge_data():

    if not os.path.exists(scraped_file):
        print("No new scraped data found.")
        return

    df = pd.read_csv(scraped_file)

    df = df.dropna(subset=["Selftext"])

    df["Author"].fillna("Anonymous", inplace=True)

    print("Processing new data...")
    df["New_Token_Selftext"] = df["Selftext"].apply(clean_and_tokenize)
    df["Lemmatized_Text"] = df["New_Token_Selftext"].apply(lemmatize_text)

    if os.path.exists(processed_output_file):
        existing_df = pd.read_csv(processed_output_file)
        print("Merging with existing processed dataset...")

        final_df = pd.concat([existing_df, df], ignore_index=True)
        final_df.drop_duplicates(subset=["Title", "Selftext"], keep="last", inplace=True)
    else:
        print("No existing processed file found. Creating a new one...")
        final_df = df

    final_df.to_csv(processed_output_file, index=False, quoting=csv.QUOTE_MINIMAL)  # Ensures words remain quoted

    print(f"Processed and merged data saved successfully as {processed_output_file}!")

clean_process_and_merge_data()

In [ ]:
import pandas as pd
import os

merged_tokenized_file = "/content/merged_output_tokenized.csv"

def check_merged_tokenized_data():
    if os.path.exists(merged_tokenized_file):
        df = pd.read_csv(merged_tokenized_file)

        row_count = df.shape[0]

        column_count = df.shape[1]

        print(f"Merged Tokenized Data Count (Rows): {row_count}")
        print(f"Total Column Count: {column_count}")

        print("\nColumn Names and Non-Null Counts:")
        print(df.count())

    else:
        print("Merged output tokenized file not found.")

check_merged_tokenized_data()

In [ ]:
import pandas as pd

file_path = "/content/merged_output_tokenized.csv"
df = pd.read_csv(file_path)

df.head(10)